# 01. DataFrame 기본 조작

`users.csv`/`orders.csv`를 명시적 스키마로 읽고 select/filter/show, printSchema를 실습하고 `collect()`와 `show()`/`take(n)`의 차이를 실행 시간으로 비교한다.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

spark = SparkSession.builder.appName("01_dataframe_basics").getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/27 01:09:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
users_schema = StructType([
    StructField("user_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("country", StringType(), True),
])
orders_schema = StructType([
    StructField("order_id", IntegerType(), False),
    StructField("user_id", IntegerType(), False),
    StructField("amount", DoubleType(), False),
])

users = spark.read.csv("/opt/spark-data/users.csv", header=True, schema=users_schema)
orders = spark.read.csv("/opt/spark-data/orders.csv", header=True, schema=orders_schema)

users.printSchema()
orders.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- country: string (nullable = true)

root
 |-- order_id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- amount: double (nullable = true)



In [3]:
users.select("user_id", "country").filter(users.country == "KR").show(5)

+-------+-------+
|user_id|country|
+-------+-------+
|      1|     KR|
|      2|     KR|
|      7|     KR|
|      9|     KR|
|     12|     KR|
+-------+-------+
only showing top 5 rows



In [4]:
import time

start = time.time()
all_orders = orders.collect()
print(f"collect() rows={len(all_orders)} elapsed={time.time() - start:.2f}s")

start = time.time()
sample = orders.take(5)
print(f"take(5) rows={len(sample)} elapsed={time.time() - start:.2f}s")

collect() rows=200000 elapsed=0.55s
take(5) rows=5 elapsed=0.04s


`collect()`는 모든 파티션의 데이터를 Driver JVM 힙 하나로 모은다. 지금은 20만 행이라 안전하지만, 실무 데이터(수억 행)에서는 Driver OOM으로 이어진다. `show()`/`take(n)`은 필요한 만큼만 Driver로 가져오므로 안전하다.

In [5]:
spark.stop()